# 01. Подготовка данных Home Credit Default Risk

**Единица анализа — одна текущая заявка `SK_ID_CURR`.** Этот notebook выполняет аудит, технический прогон и полную детерминированную подготовку данных. Модели здесь не обучаются.

Основание: локальный `Aubakirov_Dagystanov_A2_Methods_v2.pdf`, Methods and Materials — Assignment 2, особенно стр. 2. Принятый план: **7 моделей × 4 уровня × 5 повторов = 140 основных обучений**. Для каждой из 28 пар «модель + уровень» отдельно будет выполнен Randomized Search: 20 конфигураций × stratified 3-fold CV, выбор по log-loss; затем параметры фиксируются на пять повторов. Это 1 680 CV-обучений кандидатов сверх 140 основных обучений (без учёта внутренних обучений ансамблей, refit и калибровки).

Преобразования, требующие оценки по данным — медианы, масштабирование, one-hot/иное обучаемое кодирование, отбор признаков — **остаются внутри Pipeline** следующего этапа. На Home Credit здесь не вызывается `.fit()`, не выполняются ресемплинг классов и подбор признаков по TARGET. Проверка поведения будущего Pipeline использует только искусственные данные.

Исходники реализации находятся в `src/data_audit.py` и `src/data_preparation.py`; notebook явно запускает каждый этап и сохраняет реальные результаты. CSV читаются без изменений. Окружение зафиксировано в `requirements.txt` (`python -m pip install -r requirements.txt`); из корня проекта можно выполнить `python scripts/execute_notebook.py` или Jupyter Run All.

In [1]:
from pathlib import Path
import sys, os, time, json, gc, shutil
import pandas as pd
import psutil
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
assert (ROOT / 'src/data_preparation.py').is_file(), 'Запускайте notebook из корня проекта или notebooks/'
sys.path.insert(0, str(ROOT))
from src.data_audit import audit_sources
from src.data_preparation import (source_manifest, connect, run_preparation,
                                  compare_pilot_to_full, dump_json)

DATA_DIR = Path(os.environ.get('HOME_CREDIT_DATA_DIR', r'C:\Users\askat\Study\RMT\home-credit-default-risk'))
REPORTS = ROOT / 'reports'
PILOT_DIR = ROOT / 'data/processed/pilot'
FULL_DIR = ROOT / 'data/processed/full'
PILOT_SIZE = 3000
RUN_FULL = True
RUN_START = time.perf_counter()
REPORTS.mkdir(exist_ok=True)
print('Исходные CSV:', DATA_DIR)
print('RAM всего/доступно, GiB:', round(psutil.virtual_memory().total/1024**3,2),
      round(psutil.virtual_memory().available/1024**3,2))
print('Свободный диск, GiB:', round(shutil.disk_usage(ROOT).free/1024**3,2))
print('DuckDB: memory_limit=2GB; threads=2; последовательные источники, spill на диск.')

Исходные CSV: C:\Users\askat\Study\RMT\home-credit-default-risk
RAM всего/доступно, GiB: 15.82 6.58
Свободный диск, GiB: 112.7
DuckDB: memory_limit=2GB; threads=2; последовательные источники, spill на диск.


## 1. Наличие файлов и неизменность источников

Проверяются все восемь обязательных файлов. При отсутствии хотя бы одного выполнение останавливается с его именем. Для каждого сохраняется SHA-256; после подготовки хэши будут вычислены повторно. `application_test.csv` и `sample_submission.csv` не входят в исследуемую размеченную популяцию и не используются.

Сырые CSV, Parquet и временные файлы исключены через `.gitignore`. В Git допустимы только код, notebook и небольшие обезличенные сводки.

In [2]:
manifest_before = source_manifest(DATA_DIR)
dump_json(REPORTS / 'source_sha256_before.json', manifest_before)
display(pd.DataFrame(manifest_before)[['file','bytes']])

,file,bytes
0,application_train.csv,166133370
1,bureau.csv,170016717
2,bureau_balance.csv,375592889
3,POS_CASH_balance.csv,392703158
4,credit_card_balance.csv,424582605
5,installments_payments.csv,723118349
6,previous_application.csv,404973293
7,HomeCredit_columns_description.csv,37383


## 2. Полный аудит до построения признаков

Аудит читает **все строки каждого CSV**, даже если последующий технический прогон использует 3 000 клиентов. Типы сначала определяет DuckDB на 100 000 строках, затем всё содержимое читается строго: `ignore_errors=false`, несовместимый тип вызовет ошибку. Для каждого столбца сохраняются тип, число/доля пропусков, min/max числовых значений; для дат — положительные, нулевые значения и заглушки `365243`.

Проверяются точные повторы полных строк и кандидатных ключей. Для таблицы платежей повтор ключа может означать частичную оплату: автоматического удаления нет. Уникальность application, bureau и previous_application — обязательное условие соединений. История клиентов без размеченной текущей заявки учитывается в аудите, но не включается в X.

Словарь читается в `cp1252`, имена очищаются от пробелов. Документированный `SK_BUREAU_ID` сопоставляется с реальным `SK_ID_BUREAU`. Поля, описанные в словаре, но отсутствующие в CSV, не создаются. Аудит связей различает отсутствующего родителя, неверного владельца договора и клиента вне `application_train`.

In [3]:
audit_connection = connect(ROOT / 'tmp/audit_spill')
try:
    audit = audit_sources(audit_connection, DATA_DIR, REPORTS / 'audit')
finally:
    audit_connection.close()
table_audit = pd.read_csv(REPORTS / 'audit/table_summary.csv')
display(table_audit)
display(pd.read_csv(REPORTS / 'audit/target_balance.csv'))
key_tables = table_audit[table_audit.table.isin(['application_train.csv','bureau.csv','previous_application.csv'])]
assert (key_tables.key_duplicate_surplus == 0).all()
assert (key_tables.rows_with_null_key == 0).all()
assert not audit['dictionary_unmatched_columns']
print('Все реальные столбцы сопоставлены со словарём; ключи родительских таблиц уникальны.')

Аудит application_train.csv: 307,511 строк, 122 столбцов; полных дублей 0


Аудит bureau.csv: 1,716,428 строк, 17 столбцов; полных дублей 0


Аудит bureau_balance.csv: 27,299,925 строк, 3 столбцов; полных дублей 0


Аудит POS_CASH_balance.csv: 10,001,358 строк, 8 столбцов; полных дублей 0


Аудит credit_card_balance.csv: 3,840,312 строк, 23 столбцов; полных дублей 0


Аудит installments_payments.csv: 13,605,401 строк, 8 столбцов; полных дублей 0


Аудит previous_application.csv: 1,670,214 строк, 37 столбцов; полных дублей 0


,table,rows,columns,size_bytes,exact_duplicate_rows,candidate_key,key_groups,key_duplicate_surplus,max_rows_per_key,rows_with_null_key,key_expected_unique,key_note,audit_seconds
0,application_train.csv,307511,122,166133370,0,SK_ID_CURR,307511,0,1,0,True,NaN,7.741
1,bureau.csv,1716428,17,170016717,0,SK_ID_BUREAU,1716428,0,1,0,True,NaN,3.901
2,bureau_balance.csv,27299925,3,375592889,0,"SK_ID_BUREAU, MONTHS_BALANCE",27299925,0,1,0,True,NaN,14.198
3,POS_CASH_balance.csv,10001358,8,392703158,0,"SK_ID_PREV, MONTHS_BALANCE",10001358,0,1,0,True,NaN,8.799
4,credit_card_balance.csv,3840312,23,424582605,0,"SK_ID_PREV, MONTHS_BALANCE",3840312,0,1,0,True,NaN,8.707
5,installments_payments.csv,13605401,8,723118349,0,"SK_ID_PREV, NUM_INSTALMENT_VERSION, NUM_INSTAL...",13604631,770,2,2905,False,Частичные платежи могут повторять этот кандида...,12.975
6,previous_application.csv,1670214,37,404973293,0,SK_ID_PREV,1670214,0,1,0,True,NaN,6.844
7,HomeCredit_columns_description.csv,219,5,37383,0,"Table, Row",219,0,1,0,True,Словарь прочитан в кодировке cp1252,NaN


,TARGET,rows,fraction
0,0,282686,0.919271
1,1,24825,0.080729


Все реальные столбцы сопоставлены со словарём; ключи родительских таблиц уникальны.


In [4]:
column_audit = pd.read_csv(REPORTS / 'audit/column_audit.csv')
display(column_audit.sort_values('missing_fraction', ascending=False)[
    ['table','column','dtype','missing_count','missing_fraction']].head(25))
display(column_audit[column_audit.positive_count.notna()][
    ['table','column','min','max','positive_count','zero_count','placeholder_365243_count']])
relationships = pd.read_csv(REPORTS / 'audit/relationships.csv')
display(relationships)
print('Расхождения словаря:')
display(pd.read_csv(REPORTS / 'audit/dictionary_extra_rows.csv')[['Table','Row','Description']])

,table,column,dtype,missing_count,missing_fraction
194,previous_application.csv,RATE_INTEREST_PRIMARY,DOUBLE,1664263,0.996437
195,previous_application.csv,RATE_INTEREST_PRIVILEGED,DOUBLE,1664263,0.996437
138,bureau.csv,AMT_ANNUITY,DOUBLE,1226791,0.714735
76,application_train.csv,COMMONAREA_MEDI,DOUBLE,214865,0.698723
48,application_train.csv,COMMONAREA_AVG,DOUBLE,214865,0.698723
62,application_train.csv,COMMONAREA_MODE,DOUBLE,214865,0.698723
56,application_train.csv,NONLIVINGAPARTMENTS_AVG,DOUBLE,213514,0.694330
84,application_train.csv,NONLIVINGAPARTMENTS_MEDI,DOUBLE,213514,0.694330
70,application_train.csv,NONLIVINGAPARTMENTS_MODE,DOUBLE,213514,0.694330
86,application_train.csv,FONDKAPREMONT_MODE,VARCHAR,210295,0.683862


,table,column,min,max,positive_count,zero_count,placeholder_365243_count
17,application_train.csv,DAYS_BIRTH,-25229.0,-7489.0,0.0,0.0,0.0
18,application_train.csv,DAYS_EMPLOYED,-17912.0,365243.0,55374.0,2.0,55374.0
19,application_train.csv,DAYS_REGISTRATION,-24672.0,0.0,0.0,80.0,0.0
20,application_train.csv,DAYS_ID_PUBLISH,-7197.0,0.0,0.0,16.0,0.0
95,application_train.csv,DAYS_LAST_PHONE_CHANGE,-4292.0,0.0,0.0,37672.0,0.0
126,bureau.csv,DAYS_CREDIT,-2922.0,0.0,0.0,25.0,0.0
128,bureau.csv,DAYS_CREDIT_ENDDATE,-42060.0,31199.0,602603.0,883.0,0.0
129,bureau.csv,DAYS_ENDDATE_FACT,-42023.0,0.0,0.0,64.0,0.0
137,bureau.csv,DAYS_CREDIT_UPDATE,-41947.0,372.0,17.0,605.0,0.0
140,bureau_balance.csv,MONTHS_BALANCE,-96.0,0.0,0.0,610965.0,0.0


,table,check,rows_or_ids,interpretation
0,bureau.csv,rows_outside_labeled_applications,251103,Ожидаемо возможны клиенты без опубликованного ...
1,bureau.csv,distinct_clients_outside_labeled_applications,42320,Не означает сиротский ключ в полном конкурсе; ...
2,bureau.csv,rows_linked_to_labeled_applications,1465325,История с владельцем из application_train
3,POS_CASH_balance.csv,rows_outside_labeled_applications,1457983,Ожидаемо возможны клиенты без опубликованного ...
4,POS_CASH_balance.csv,distinct_clients_outside_labeled_applications,47808,Не означает сиротский ключ в полном конкурсе; ...
5,POS_CASH_balance.csv,rows_linked_to_labeled_applications,8543375,История с владельцем из application_train
6,credit_card_balance.csv,rows_outside_labeled_applications,612347,Ожидаемо возможны клиенты без опубликованного ...
7,credit_card_balance.csv,distinct_clients_outside_labeled_applications,16653,Не означает сиротский ключ в полном конкурсе; ...
8,credit_card_balance.csv,rows_linked_to_labeled_applications,3227965,История с владельцем из application_train
9,installments_payments.csv,rows_outside_labeled_applications,2013809,Ожидаемо возможны клиенты без опубликованного ...


Расхождения словаря:


,Table,Row,Description
0,previous_application.csv,NFLAG_MICRO_CASH,Flag Micro finance loan


## 3. Временная доступность и правила признаков

В датасете нет календарных дат и полного журнала изменений. Поэтому правила ниже обеспечивают воспроизводимое отсечение по имеющимся полям, но не доказывают point-in-time происхождение каждого атрибута. Анкета, включая `EXT_SOURCE_1/2/3`, трактуется как снимок при подаче заявки; словарь не раскрывает даты внешних оценок. Финальные состояния, для которых нельзя обосновать дату доступности, не используются.

| Источник | Правило и смысл |
|---|---|
| application | `DAYS_EMPLOYED=365243` и другие положительные фактические даты превращаются в NULL; для неизвестной даты трудоустройства есть индикатор. Возраст и пол удаляются из X до создания отношений. |
| bureau | `DAYS_CREDIT < 0` и `DAYS_CREDIT_UPDATE < 0`: решение и последняя информация строго до дня заявки. `DAYS_CREDIT_ENDDATE` — оставшаяся договорная длительность, поэтому положительные плановые сроки сохраняются. `DAYS_ENDDATE_FACT` допускается только <0. |
| bureau_balance | `MONTHS_BALANCE < 0`, затем связь через допустимые bureau → application. Месяц 0 исключён: точный день снимка неизвестен. |
| POS / credit card | `MONTHS_BALANCE < 0`, прямая связь по `SK_ID_CURR`. Словарь POS разрешает месяц 0 как снимок при заявке, но для единообразного строгого режима он не используется. Количество будущих взносов — известное в историческом снимке плановое обязательство. |
| installments | Фактический `DAYS_ENTRY_PAYMENT < 0`; неизвестные/будущие фактические платежи не используются. Положительный `DAYS_INSTALMENT` не удаляется автоматически: это может быть срок уже совершённой предоплаты. `365243` не является датой. |
| previous | Только `DAYS_DECISION < 0`. `DAYS_FIRST_DUE` и `DAYS_LAST_DUE_1ST_VERSION` — договорные даты, положительные сохраняются, `365243` → NULL. `DAYS_FIRST_DRAWING`, `DAYS_LAST_DUE`, `DAYS_TERMINATION` исключены из X из-за неоднозначности доступности/обновлений. |

Связь `SK_ID_PREV` с previous проверяется аудитом. Отсутствие опубликованной предыдущей заявки не отменяет корректную историю с прямым `SK_ID_CURR` и допустимой собственной датой; Level 3 не зависит от наличия таблицы Level 4. Все исключения считаются и сохраняются в `temporal_filters.csv`; конечный объём — в `history_coverage.csv`.

Полные дубли диагностируются, но без идентификатора события не удаляются вслепую. По взносам сначала суммируются частичные `AMT_PAYMENT` внутри `(SK_ID_CURR, SK_ID_PREV, NUM_INSTALMENT_VERSION, NUM_INSTALMENT_NUMBER)`. Предписанная сумма берётся один раз. Если внутри ключа различаются срок или сумма, этот взнос исключается из агрегатов, число таких случаев сохраняется. Доля просрочки относится **к наблюдаемым оплаченным взносам**; полностью отсутствующие неоплаченные обязательства из этой таблицы не восстанавливаются.

Суммы взносов и частичных оплат агрегируются как **DECIMAL(24,3)**: у реального CSV проверены максимум три десятичных знака. Это исключает ложную недоплату из-за порядка сложения floating-point значений на границе равенства. Перед преобразованием проверяется отсутствие большей точности; финальные числовые признаки сохраняются в обычном числовом формате. Просрочка по дате последнего платежа — наблюдаемое приближение: поздняя переплата может пометить взнос, полностью погашенный вовремя, как поздний. Это не точная дата погашения долга. Нулевой или отрицательный знаменатель отношения даёт NULL, а не бесконечность. Mean/min/max историй описывают типичный уровень и диапазон; средние по месячным таблицам взвешены числом наблюдаемых договор-месяцев. Индикаторы `HAS_*` означают наличие пригодных агрегатов. При отсутствии истории они равны 0, а суммы, доли и количества остаются NULL.

## 4. Четыре вложенных уровня и технический прогон

1. **Level 1:** application и пять отношений (credit/income, annuity/income, credit/goods, annuity/credit, income/family), индикатор неизвестной даты трудоустройства.
2. **Level 2:** Level 1 + bureau и bureau_balance: число кредитов, задолженность, просрочки, сроки и доли месячных статусов.
3. **Level 3:** Level 2 + POS и card: история взносов и просрочек, обороты, платежи и использование карточного лимита.
4. **Level 4:** Level 3 + installments и previous: оплата/просрочка наблюдаемых взносов, суммы и исходы предыдущих заявок.

Выбираем 3 000 **клиентов** стратифицированно с seed 2026 и читаем всю связанную с ними историю, включая bureau_balance через bureau. Нельзя брать первые строки каждой таблицы независимо. Технический набор сохраняется отдельно и не заменяет полный результат. Каждый этап сначала сохраняет агрегат с уникальным `SK_ID_CURR`, затем используется left join; размножение строк запрещено assertions.

In [5]:
pilot = run_preparation(DATA_DIR, PILOT_DIR, REPORTS / 'pilot', sample_size=PILOT_SIZE)
display(pd.DataFrame(pilot['shapes']))
display(pd.DataFrame(pilot['history_coverage']))
print('Пилот: все проверки пройдены:', pilot['checks'])
print('Время пилота, секунд:', pilot['elapsed_seconds'])

application: 3,000 клиентов, 124 признаков; 1.6 с


bureau: 2,569 клиентов, 42 признаков; 0.9 с


bb: 897 клиентов, 10 признаков; 1.8 с


pos: 2,795 клиентов, 19 признаков; 1.6 с


cc: 841 клиентов, 65 признаков; 2.0 с


...

Level 3: 3,000 x 264; 0.8 MiB


Level 4: 3,000 x 322; 1.2 MiB


,level,rows,features,bytes
0,1,3000,124,362594
1,2,3000,178,599289
2,3,3000,264,858570
3,4,3000,322,1307938


,table,cohort_raw_rows,eligible_rows,eligible_clients,eligible_client_share
0,bureau,14236,14232,2569,0.856333
1,bb,153654,150195,897,0.299000
2,pos,81734,81734,2795,0.931667
3,cc,33501,33501,841,0.280333
4,installments,111993,111932,2822,0.940667
5,previous,13499,13499,2815,0.938333


Пилот: все проверки пройдены: {'unique_applicants': True, 'same_clients_all_levels': True, 'nested_features_and_equal_values': True, 'no_target_ids_gender_age_in_X': True, 'no_train_test_overlap': True, 'repeats_subset_of_development': True, 'no_infinite_values': True, 'missing_history_remains_null': True}
Время пилота, секунд: 52.533


## 5. Полная подготовка и сохранённые разбиения

Полная подготовка запускается только после успешного пилота и проверки ресурсов. DuckDB ограничен 2 GB и может выгружать промежуточные операции на диск. В памяти одновременно не удерживаются все исторические таблицы и четыре X. Для старта требуется ≥3 GiB доступной RAM и ≥10 GiB свободного диска; при нехватке notebook явно запишет причину и оставшееся действие.

Разбиение выполняется один раз для полной популяции: `train_test_split(test_size=0.20, stratify=TARGET, random_state=42)`. Development служит для последующего поиска; test сохраняется для итоговой оценки. Пять 90%-ных стратифицированных подвыборок development используют seeds 42…46. Оставшиеся 10% каждого повтора не становятся новым test. Подвыборки могут пересекаться между собой — это ожидается. Дополнительно сохраняется общий 3-fold для 28 будущих поисков. После сохранения следует загружать эти ID, а не генерировать разбиения отдельно по уровням/моделям.

Fairness хранится отдельно: исходные `CODE_GENDER`, `DAYS_BIRTH`, `AGE_YEARS=-DAYS_BIRTH/365.25`, группы `<25`/`>=25`, неизвестный пол отдельной группой. Возраст в годах приближённый: календарных дат нет. Ни эти значения, ни ID, ни TARGET не входят в X. Исключение прямых защищённых атрибутов не гарантирует отсутствия косвенных прокси.

In [6]:
available_gib = psutil.virtual_memory().available / 1024**3
disk_gib = shutil.disk_usage(ROOT).free / 1024**3
full = None
full_skip_reason = None
if not RUN_FULL:
    full_skip_reason = 'RUN_FULL=False: для завершения установить True и выполнить этапы 5–7.'
elif available_gib < 3 or disk_gib < 10:
    full_skip_reason = f'Недостаточно ресурсов: RAM={available_gib:.2f} GiB, диск={disk_gib:.2f} GiB. Освободить ресурсы и повторить этапы 5–7.'
else:
    full = run_preparation(DATA_DIR, FULL_DIR, REPORTS / 'full', sample_size=None)
if full is None:
    print('ПОЛНЫЙ ЗАПУСК НЕ ВЫПОЛНЕН:', full_skip_reason)
else:
    print('ПОЛНЫЙ ЗАПУСК ВЫПОЛНЕН:', full['n_applicants'], 'текущих заявок')
    display(pd.DataFrame(full['shapes']))
    display(pd.DataFrame(full['splits']))
    display(pd.DataFrame(full['history_coverage']))
    print('Противоречивых взносов исключено:', full['installment_conflicting_obligations_excluded'])

application: 307,511 клиентов, 124 признаков; 3.5 с


bureau: 263,475 клиентов, 42 признаков; 2.1 с


bb: 92,039 клиентов, 10 признаков; 3.8 с


pos: 289,444 клиентов, 19 признаков; 3.5 с


cc: 86,905 клиентов, 65 признаков; 3.8 с


...

ПОЛНЫЙ ЗАПУСК ВЫПОЛНЕН: 307511 текущих заявок


,level,rows,features,bytes
0,1,307511,124,22888092
1,2,307511,178,43204706
2,3,307511,264,63449031
3,4,307511,322,103699046


,split,n,target_0,target_1,positive_rate
0,development,246008,226148,19860,0.080729
1,test,61503,56538,4965,0.080728
2,train_seed_42,221407,203533,17874,0.080729
3,train_seed_43,221407,203533,17874,0.080729
4,train_seed_44,221407,203533,17874,0.080729
5,train_seed_45,221407,203533,17874,0.080729
6,train_seed_46,221407,203533,17874,0.080729


,table,cohort_raw_rows,eligible_rows,eligible_clients,eligible_client_share
0,bureau,1465325,1464821,263475,0.856799
1,bb,14701612,14353734,92039,0.299303
2,pos,8543375,8543375,289444,0.941248
3,cc,3227965,3227965,86905,0.282608
4,installments,11591592,11589009,291635,0.948373
5,previous,1413701,1413701,291057,0.946493


Противоречивых взносов исключено: 0


## 6. Проверки результата и формат файлов

`X_level_N.parquet` содержит **только признаки**, без идентификаторов даже в индексе. Строки всех четырёх файлов упорядочены по `SK_ID_CURR`; их точное соответствие хранится в `row_ids.parquet`. `target.parquet` и `fairness.parquet` содержат ID для безопасного совмещения. JSON-списки фиксируют порядок и вложенность признаков, а `feature_catalog.csv` сохраняет смысл и SQL-формулу каждого признака.

Автоматические проверки: уникальность заявки и ключей агрегатов; одинаковые клиенты и количество строк на всех уровнях; равенство общих признаков; отсутствие TARGET/ID/пола/возраста в X; непересечение development/test; принадлежность повторов development; отсутствие бесконечностей; NULL у отсутствующей истории. Ни один клиент не отбрасывается из-за отсутствующей истории.

Дополнительная проверка сравнивает значения пилота и полного результата для **тех же клиентов**, читая только 30 столбцов за раз. Это выявляет ошибочный выбор только части истории и зависимость детерминированных признаков от состава выборки. Допуск `1e-10` нужен для порядка суммирования floating-point агрегатов при параллельной обработке; это не обучение параметров.

In [7]:
crosscheck = compare_pilot_to_full(PILOT_DIR, FULL_DIR) if full is not None else None
print('Сравнение пилота с полным набором:', crosscheck)
manifest_after = source_manifest(DATA_DIR)
assert manifest_after == manifest_before, 'Один из исходных CSV изменён!'
dump_json(REPORTS / 'source_sha256_after.json', manifest_after)
print('Все восемь исходных CSV неизменны: SHA-256 совпадают.')
dump_json(REPORTS / 'run_summary.json', {
    'processed_scope': 'full' if full is not None else 'technical_sample_only',
    'audit_seconds': audit['elapsed_seconds'], 'pilot': pilot, 'full': full,
    'full_skip_reason': full_skip_reason, 'pilot_full_comparison': crosscheck,
    'original_csv_sha256_unchanged': True,
    'total_elapsed_seconds': round(time.perf_counter()-RUN_START,3)
})
print('Проверки:', (full or pilot)['checks'])

Сравнение пилота с полным набором: {'pilot_clients': 3000, 'levels_compared': 4, 'same_values': True, 'tolerance': 1e-10}


Все восемь исходных CSV неизменны: SHA-256 совпадают.
Проверки: {'unique_applicants': True, 'same_clients_all_levels': True, 'nested_features_and_equal_values': True, 'no_target_ids_gender_age_in_X': True, 'no_train_test_overlap': True, 'repeats_subset_of_development': True, 'no_infinite_values': True, 'missing_history_remains_null': True}


## 7. Итог и передача на этап моделирования

Следующая ячейка показывает фактически выполненный объём, размеры, баланс, время и перечень файлов. Большие данные находятся в `data/processed/full/`; промежуточные агрегаты — в `aggregates/`; ID — в `splits/`. Пилот находится в соседней папке `pilot/`. Небольшие отчёты остаются в `reports/`.

Чтобы восстановить X по ID на следующем этапе:

```python
ids = pd.read_parquet(FULL_DIR / 'row_ids.parquet')['SK_ID_CURR']
X = pd.read_parquet(FULL_DIR / 'X_level_1.parquet')
X.index = pd.Index(ids, name='SK_ID_CURR')  # ID служит индексом, не входным столбцом
y = pd.read_parquet(FULL_DIR / 'target.parquet').set_index('SK_ID_CURR')['TARGET']
dev_ids = pd.read_parquet(FULL_DIR / 'splits/development_ids.parquet')['SK_ID_CURR']
X_dev, y_dev = X.loc[dev_ids], y.loc[dev_ids]
```

Fit медиан, индикаторов пропусков, категории и стандартизация выполняются **внутри каждого CV-train**. После поиска они заново fit на каждой 90%-ной train-подвыборке с уже фиксированными гиперпараметрами. Test не используется для этих решений; fairness-файл доступен только для предусмотренного протокола анализа/коррекции. Для всех моделей внутри уровня используется один и тот же список исходных признаков.

In [8]:
result = full or pilot
print('Реально обработано:', result['scope'], '—', result['n_applicants'], 'заявок')
display(pd.DataFrame(result['shapes']).rename(columns={'features':'n_features','rows':'n_rows'}))
display(pd.DataFrame([{'TARGET=0': result['target_0'], 'TARGET=1': result['target_1'],
                       'доля TARGET=1': result['positive_rate']}]))
run_report = json.loads((REPORTS/'run_summary.json').read_text(encoding='utf-8'))
print('Время, секунд: аудит=', audit['elapsed_seconds'], 'пилот=', pilot['elapsed_seconds'],
      'полный=', full['elapsed_seconds'] if full else None,
      'весь notebook=', run_report['total_elapsed_seconds'])
generated = []
for directory in [ROOT/'data/processed', REPORTS, ROOT/'notebooks', ROOT/'src', ROOT/'scripts', ROOT/'tests']:
    for path in sorted(directory.rglob('*')):
        if path.is_file() and 'duckdb_tmp' not in str(path) and '__pycache__' not in str(path):
            generated.append({'file':str(path.relative_to(ROOT)), 'bytes':path.stat().st_size})
for name in ['.gitignore','requirements.txt']:
    path = ROOT/name
    generated.append({'file':name, 'bytes':path.stat().st_size})
inventory = pd.DataFrame(generated)
inventory.to_csv(REPORTS/'generated_files.csv', index=False)
with pd.option_context('display.max_rows', None, 'display.max_colwidth', 120):
    display(inventory)
if full_skip_reason:
    print(full_skip_reason)

Реально обработано: full — 307511 заявок


,level,n_rows,n_features,bytes
0,1,307511,124,22888092
1,2,307511,178,43204706
2,3,307511,264,63449031
3,4,307511,322,103699046


,TARGET=0,TARGET=1,доля TARGET=1
0,282686,24825,0.080729


Время, секунд: аудит= 97.229 пилот= 52.533 полный= 151.576 весь notebook= 309.197


,file,bytes
0,data\processed\full\aggregates\application.parquet,23202644
1,data\processed\full\aggregates\bb.parquet,1531011
2,data\processed\full\aggregates\bureau.parquet,17255656
3,data\processed\full\aggregates\bureau_id_map.parquet,3310441
4,data\processed\full\aggregates\cc.parquet,9953044
5,data\processed\full\aggregates\installment_obligations.parquet,196304476
6,data\processed\full\aggregates\installments.parquet,13833955
7,data\processed\full\aggregates\pos.parquet,5036909
8,data\processed\full\aggregates\previous.parquet,25911140
9,data\processed\full\fairness.parquet,3006562


## 8. Дополнительная проверка качества перед моделированием

Этот раздел использует **существующие** `development_ids.parquet`, четыре X и списки признаков. Разбиения не создаются повторно, признаки не перезаписываются. SHA-256 всех подготовленных файлов проверяется до и после. Распределения, процентили и смысловые ограничения анализируются только на **development**; test участвует лишь в механических проверках сохранности и в запрошенной разбивке 2 905 строк с неполным ключом. Решения о выбросах по test не принимаются.

Результаты записываются в `reports/quality/`. Число **значений исходной истории**, **клиентов** и **ячеек агрегатов** указывается раздельно: это разные единицы счёта.

In [9]:
from pathlib import Path
import sys, os, json, time
import pandas as pd
from IPython.display import display

QA_ROOT=Path.cwd()
if QA_ROOT.name=='notebooks':
    QA_ROOT=QA_ROOT.parent
sys.path.insert(0,str(QA_ROOT))
from src.data_quality import (artifact_manifest, development_profile, semantic_checks, sentinel_checks)
from src.payment_quality import audit_payment_keys
from src.fold_preprocessing import make_numeric_pipeline, numeric_pipeline_smoke_check
from src.data_preparation import dump_json, source_manifest

QA_DATA=Path(os.environ.get('HOME_CREDIT_DATA_DIR',r'C:\Users\askat\Study\RMT\home-credit-default-risk'))
QA_PREP=QA_ROOT/'data/processed/full'
QA_REPORT=QA_ROOT/'reports/quality'
QA_REPORT.mkdir(parents=True,exist_ok=True)
QA_STARTED=time.perf_counter()
qa_manifest_before=artifact_manifest(QA_PREP)
qa_sources_before=source_manifest(QA_DATA)
dump_json(QA_REPORT/'prepared_sha256_before.json',qa_manifest_before)
print('Существующих подготовленных файлов под защитой SHA-256:',len(qa_manifest_before))

Существующих подготовленных файлов под защитой SHA-256: 29


### 8.1. Числовые распределения development

Вычисляются точные линейно интерполированные P1, P25, медиана, P75, P99, P99,9 по непустым значениям; отдельно — пропуски, min/max, нули, отрицательные значения и бесконечности. Вычисление медианы для **отчёта** не заполняет данные и не обучает преобразование. Читается по 24 столбца, чтобы не держать в памяти все уровни.

Полная таблица — `numeric_development_summary.csv`; `first_level` и `levels` показывают принадлежность признака уровням. Общие признаки профилируются один раз. Отдельно различаются полностью пустой столбец, один уникальный **наблюдаемый** уровень при наличии пропусков и истинно постоянный столбец без пропусков. Ничего не удаляется по этой глобальной сводке.

In [10]:
qa_profile=development_profile(QA_PREP,QA_REPORT)
display(pd.DataFrame(qa_profile['levels']))
numeric_summary=pd.read_csv(QA_REPORT/'numeric_development_summary.csv')
print('Полная сводка:',QA_REPORT/'numeric_development_summary.csv')
display(numeric_summary.sort_values('missing_fraction',ascending=False)[
    ['feature','first_level','missing','missing_fraction','min','p01','p25','median','p75','p99','p999','max','zeros','negatives']].head(25))
print('Пустые и постоянные столбцы (все типы):')
display(pd.read_csv(QA_REPORT/'empty_and_constant_columns.csv'))
print('Полностью пустые числовые:',qa_profile['all_missing_numeric'])
print('Истинно постоянные числовые без пропусков:',qa_profile['constant_fully_numeric'])

,level,rows,total_features,numeric_features,all_missing_numeric,constant_observed_numeric
0,1,246008,124,109,0,0
1,2,246008,178,163,0,0
2,3,246008,264,249,0,2
3,4,246008,322,307,0,2


Полная сводка: C:\Users\askat\Documents\ChatGPT\RMT\reports\quality\numeric_development_summary.csv


,feature,first_level,missing,missing_fraction,min,p01,p25,median,p75,p99,p999,max,zeros,negatives
211,CC_AMT_PAYMENT_CURRENT_MAX,3,197059,0.801027,0.000000,930.1428,22500.000000,65250.000000,157500.000000,748601.411400,1.028514e+06,4.289207e+06,103,0
210,CC_AMT_PAYMENT_CURRENT_MIN,3,197059,0.801027,0.000000,0.0000,0.000000,0.000000,597.375000,27000.000000,1.002379e+05,1.125000e+06,30774,0
209,CC_AMT_PAYMENT_CURRENT_AVG,3,197059,0.801027,0.000000,248.8032,4657.500000,9855.000000,21416.544783,122128.860600,3.229143e+05,1.125000e+06,103,0
224,CC_CNT_DRAWINGS_ATM_CURRENT_AVG,3,196989,0.800742,0.000000,0.0000,0.063158,0.272727,0.733333,3.857143,8.000000e+00,3.500000e+01,7884,0
226,CC_CNT_DRAWINGS_ATM_CURRENT_MAX,3,196989,0.800742,0.000000,0.0000,2.000000,3.000000,6.000000,17.000000,2.798200e+01,5.100000e+01,7884,0
230,CC_CNT_DRAWINGS_OTHER_CURRENT_AVG,3,196989,0.800742,0.000000,0.0000,0.000000,0.000000,0.000000,0.166667,6.666667e-01,2.000000e+00,44467,0
231,CC_CNT_DRAWINGS_OTHER_CURRENT_MIN,3,196989,0.800742,0.000000,0.0000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,2.000000e+00,49016,0
232,CC_CNT_DRAWINGS_OTHER_CURRENT_MAX,3,196989,0.800742,0.000000,0.0000,0.000000,0.000000,0.000000,2.000000,4.000000e+00,1.200000e+01,44467,0
233,CC_CNT_DRAWINGS_POS_CURRENT_AVG,3,196989,0.800742,0.000000,0.0000,0.000000,0.043478,1.434783,20.544196,4.290173e+01,1.182500e+02,21640,0
234,CC_CNT_DRAWINGS_POS_CURRENT_MIN,3,196989,0.800742,0.000000,0.0000,0.000000,0.000000,0.000000,5.000000,2.000000e+01,7.800000e+01,46809,0


Пустые и постоянные столбцы (все типы):


,feature,first_level,levels,dtype,n_development,missing,missing_fraction,n_unique_observed,all_missing,constant_among_observed,constant_including_missing,constant_value
0,CC_SK_DPD_MIN,3,"3,4",float64,246008,176325,0.716745,1,False,True,False,0.0
1,CC_SK_DPD_DEF_MIN,3,"3,4",float64,246008,176325,0.716745,1,False,True,False,0.0


Полностью пустые числовые: []
Истинно постоянные числовые без пропусков: []


### 8.2. Смысловые ограничения и предлагаемые действия

Строгие правила: положительные текущие доход/кредит; неотрицательные номинальные суммы; целочисленные счётчики семьи и детей; доли `*_SHARE` в `[0,1]`. Пустое значение не считается отрицательным или нулевым. Отсутствие нарушения знака не подтверждает достоверность экстремального значения.

Отношения сумм **не обязаны лежать в `[0,1]`**: переплата, превышение лимита и отличие одобренного кредита от запрошенного допустимы. Словарь не задаёт сопоставимые периоды дохода и аннуитета — `annuity/income > 1` само по себе не основание исправления. Отрицательные остатки и денежные движения могут быть переплатой/сторно, поэтому общее правило «все AMT < 0 удалить» не применяется.

У каждого правила в `semantic_source_checks.csv` указаны SQL-условие, обоснование, действие, статус и число затронутых исходных значений/клиентов. Конкретное предложение для отрицательного первоначального взноса — заменить его NULL **до** агрегации после подтверждения знаковой конвенции; вычисляется точное число изменившихся агрегатов, но замена сейчас не применяется. Для неоднозначных лимитов/движений предлагается уточнение источника, а не неподтверждённое исправление.

Редкие большие значения сохраняются. P99/P99,9 служат описанием хвоста, а не порогами удаления, clipping или winsorization.

In [11]:
qa_semantics=semantic_checks(QA_DATA,QA_PREP,QA_REPORT)
display(pd.read_csv(QA_REPORT/'semantic_source_checks.csv')[
    ['source','field','condition','affected_values','affected_clients','classification','proposed_action_ru','reason_ru']])
display(pd.read_csv(QA_REPORT/'ratio_checks_development.csv'))
focus=['AMT_INCOME_TOTAL','AMT_CREDIT','AMT_ANNUITY','AMT_GOODS_PRICE','CNT_FAM_MEMBERS','CNT_CHILDREN',
       'APP_CREDIT_TO_INCOME','APP_ANNUITY_TO_INCOME','APP_CREDIT_TO_GOODS','APP_ANNUITY_TO_CREDIT','APP_INCOME_PER_MEMBER']
display(numeric_summary[numeric_summary.feature.isin(focus)][
    ['feature','missing','min','p01','p25','median','p75','p99','p999','max','zeros','negatives']])
print('Оценка воздействия предлагаемой замены, не применено:')
display(pd.DataFrame([qa_semantics['proposed_fix']]))
print('Хвост дохода, сохранён без изменений:',qa_semantics['income_tail'])

,source,field,condition,affected_values,affected_clients,classification,proposed_action_ru,reason_ru
0,application,AMT_INCOME_TOTAL,AMT_INCOME_TOTAL<=0,0,0,hard,NULL при доходе <=0; сейчас не требуется,Положительный доход необходим для интерпретаци...
1,application,AMT_CREDIT,AMT_CREDIT<=0,0,0,hard,NULL при кредите <=0; сейчас не требуется,Текущая выдаваемая сумма должна быть положител...
2,application,AMT_ANNUITY,AMT_ANNUITY<0,0,0,hard,NULL для отрицательного аннуитета; сейчас не т...,Отрицательная сумма регулярного обязательства ...
3,application,AMT_GOODS_PRICE,AMT_GOODS_PRICE<0,0,0,hard,NULL для отрицательной стоимости; сейчас не тр...,Стоимость приобретаемого товара неотрицательна.
4,application,CNT_FAM_MEMBERS,CNT_FAM_MEMBERS<1 OR CNT_FAM_MEMBERS<>floor(CN...,0,0,hard,NULL при числе членов семьи <1 или нецелом; се...,Число членов семьи включает заявителя и являет...
5,application,CNT_CHILDREN,CNT_CHILDREN<0 OR CNT_CHILDREN<>floor(CNT_CHIL...,0,0,hard,NULL для отрицательного/дробного счётчика; сей...,Число детей — неотрицательное целое.
6,application,CNT_FAM_MEMBERS/CNT_CHILDREN,CNT_FAM_MEMBERS<=CNT_CHILDREN,0,0,review,"Проверить состав семьи, не исправлять автомати...",Условие предполагает включение заявителя и дет...
7,previous,AMT_APPLICATION,AMT_APPLICATION<0,0,0,hard,NULL для отрицательной запрошенной суммы; сейч...,Номинальная запрошенная сумма неотрицательна.
8,previous,AMT_CREDIT,AMT_CREDIT<0,0,0,hard,NULL для отрицательной одобренной суммы; сейча...,Номинальная сумма кредита неотрицательна.
9,previous,AMT_APPLICATION,AMT_APPLICATION=0,260183,109383,allowed,Сохранить,Отменённые/отклонённые заявки и иной тип проду...


,feature,missing,negative,above_one,infinite,rule_violations,rule_ru,applied_correction
0,APP_CREDIT_TO_INCOME,0,0,233085,0,0,Положительное отношение сумм не обязано быть <...,False
1,APP_ANNUITY_TO_INCOME,10,0,25,0,0,Положительное отношение сумм не обязано быть <...,False
2,APP_CREDIT_TO_GOODS,221,0,158985,0,0,Положительное отношение сумм не обязано быть <...,False
3,APP_ANNUITY_TO_CREDIT,10,0,0,0,0,Положительное отношение сумм не обязано быть <...,False
4,APP_INCOME_PER_MEMBER,2,0,246006,0,0,Положительное отношение сумм не обязано быть <...,False
5,BUREAU_ACTIVE_SHARE,35256,0,0,0,0,"Доля должна лежать в [0,1].",False
6,BUREAU_CLOSED_SHARE,35256,0,0,0,0,"Доля должна лежать в [0,1].",False
7,BUREAU_DEBT_TO_CREDIT,41932,1002,1244,0,0,Отрицательный долг возможен при переплате; дол...,False
8,BB_DELINQUENT_SHARE,172496,0,0,0,0,"Доля должна лежать в [0,1].",False
9,BB_SEVERE_DELINQUENT_SHARE,172496,0,0,0,0,"Доля должна лежать в [0,1].",False


,feature,missing,min,p01,p25,median,p75,p99,p999,max,zeros,negatives
0,CNT_CHILDREN,0,0.000000,0.000000,0.000000,0.000000,1.000000,3.000000e+00,4.000000e+00,1.900000e+01,172232,0
1,AMT_INCOME_TOTAL,0,25650.000000,45000.000000,112500.000000,147600.000000,202500.000000,4.725000e+05,9.000000e+05,1.170000e+08,0,0
2,AMT_CREDIT,0,45000.000000,76410.000000,270000.000000,514777.500000,808650.000000,1.862802e+06,2.517300e+06,4.050000e+06,0,0
3,AMT_ANNUITY,10,1615.500000,6187.500000,16510.500000,24903.000000,34596.000000,7.003350e+04,1.110018e+05,2.580255e+05,0,0
4,AMT_GOODS_PRICE,221,40500.000000,67500.000000,238500.000000,450000.000000,679500.000000,1.800000e+06,2.250000e+06,4.050000e+06,0,0
16,CNT_FAM_MEMBERS,2,1.000000,1.000000,2.000000,2.000000,3.000000,5.000000e+00,6.000000e+00,2.000000e+01,0,0
104,APP_CREDIT_TO_INCOME,0,0.004808,0.595789,2.018667,3.268948,5.168514,1.303571e+01,1.930662e+01,4.922720e+01,0,0
105,APP_ANNUITY_TO_INCOME,10,0.000224,0.039371,0.114583,0.162833,0.229000,4.838502e-01,7.262502e-01,1.570600e+00,0,0
106,APP_CREDIT_TO_GOODS,221,0.150000,1.000000,1.000000,1.118800,1.198000,1.475200e+00,1.528000e+00,6.000000e+00,0,0
107,APP_ANNUITY_TO_CREDIT,10,0.022073,0.026459,0.036900,0.050000,0.063910,1.144990e-01,1.201000e-01,1.244296e-01,0,0


Оценка воздействия предлагаемой замены, не применено:


,proposal,raw_values,changed_aggregate_cells,changed_clients,per_feature_changed_cells,applied,reason
0,previous AMT_DOWN_PAYMENT<0 -> NULL before agg...,1,2,1,"{'avg': 1, 'min': 1, 'max': 0}",False,"Сначала требуется подтвердить, что -0.9 не явл..."


Хвост дохода, сохранён без изменений: {'p99': 472500.0, 'p999': 900000.0, 'max': 117000000.0, 'above_p999': 222, 'removed': 0, 'winsorized': 0, 'note': 'Квантили вычислены только на development и не являются порогами очистки.'}


### 8.3. Заглушки: источник → правило → сохранённый результат

Для `DAYS_EMPLOYED=365243` проверяется не только отсутствие числа в X, но и соответствие **каждой заявки** исходнику: значение стало NULL, `APP_EMPLOYMENT_DATE_UNKNOWN=1`, остальные допустимые даты сохранены. Аналогично проверяются остальные фактические даты анкеты.

В previous `365243` заменён NULL в используемых плановых датах `DAYS_FIRST_DUE` и `DAYS_LAST_DUE_1ST_VERSION`. Их AVG/MIN/MAX заново вычисляются из очищенных строк и сверяются с сохранёнными агрегатами для всех подготовленных заявок. `DAYS_FIRST_DRAWING`, `DAYS_LAST_DUE`, `DAYS_TERMINATION` вообще не входят в X: для них «после» означает отсутствие признака, а не число заполненных значений. Положительные плановые сроки сохраняются.

В таблице до/после NULL подсчитаны на одном наборе **исходных строк**, поскольку сравнивать число исходных NULL с числом клиентских агрегатов некорректно. Проверка агрегатов выделена в отдельный файл. `placeholder_before` везде означает только `365243`. Вспомогательный `positive_before` для фактических дат application считает все положительные значения (включая заглушку), а для плановых дат previous/bureau — только положительные значения без заглушки: эти поля проверяются по разным смысловым правилам. Общая механическая проверка итоговых временных признаков не подбирает параметры по test.

In [12]:
qa_sentinels=sentinel_checks(QA_DATA,QA_PREP,QA_REPORT)
display(pd.read_csv(QA_REPORT/'sentinels_before_after.csv')[
    ['source','field','scope','source_rows','placeholder_before','null_before','positive_before',
     'null_after_same_rows','placeholder_after_same_rows','saved_feature_status']])
display(pd.read_csv(QA_REPORT/'saved_date_aggregate_checks.csv'))
print('Проверки сохранённых заглушек:',qa_sentinels)

,source,field,scope,source_rows,placeholder_before,null_before,positive_before,null_after_same_rows,placeholder_after_same_rows,saved_feature_status
0,application,DAYS_EMPLOYED,all_source,307511,55374,0,55374,55374.0,0.0,included
1,application,DAYS_REGISTRATION,all_source,307511,0,0,0,0.0,0.0,included
2,application,DAYS_ID_PUBLISH,all_source,307511,0,0,0,0.0,0.0,included
3,application,DAYS_LAST_PHONE_CHANGE,all_source,307511,0,1,0,1.0,0.0,included
4,application,DAYS_EMPLOYED,development,246008,44143,0,44143,44143.0,0.0,included
5,application,DAYS_REGISTRATION,development,246008,0,0,0,0.0,0.0,included
6,application,DAYS_ID_PUBLISH,development,246008,0,0,0,0.0,0.0,included
7,application,DAYS_LAST_PHONE_CHANGE,development,246008,0,1,0,1.0,0.0,included
8,previous,DAYS_FIRST_DRAWING,all_source,1670214,934444,673065,0,NaN,NaN,excluded
9,previous,DAYS_FIRST_DUE,all_source,1670214,40645,673065,0,713710.0,0.0,included


,feature,prepared_clients_with_history,mismatches_to_cleaned_history,positive_saved_values,remaining_365243
0,PREVIOUS_DAYS_FIRST_DUE_AVG,291057,0,0,0
1,PREVIOUS_DAYS_FIRST_DUE_MIN,291057,0,0,0
2,PREVIOUS_DAYS_FIRST_DUE_MAX,291057,0,0,0
3,PREVIOUS_DAYS_LAST_DUE_1ST_VERSION_AVG,291057,0,46597,0
4,PREVIOUS_DAYS_LAST_DUE_1ST_VERSION_MIN,291057,0,29608,0
5,PREVIOUS_DAYS_LAST_DUE_1ST_VERSION_MAX,291057,0,147954,0


Проверки сохранённых заглушек: {'source_before_after_rows': 20, 'application_rule_mismatches': 0, 'employment_indicator_mismatches': 0, 'saved_previous_aggregate_mismatches': 0, 'development_saved_date_placeholders': 0, 'all_prepared_saved_date_placeholders': 0, 'note': 'Число NULL до/после приводится для исходных строк; проверка агрегатов сохранена отдельно. У исключённых полей после=не применимо.'}


### 8.4. Неполный ключ 2 905 строк installments_payments

Проверяемый кандидатный ключ строки платежа полностью:
`(SK_ID_PREV, NUM_INSTALMENT_VERSION, NUM_INSTALMENT_NUMBER, DAYS_ENTRY_PAYMENT)`.
`SK_ID_CURR` дополнительно задаёт владельца и связь с текущей заявкой. Это **не гарантированно уникальный ID платёжного события**: несколько частичных платежей могут иметь одну дату и один кандидатный ключ.

Ключ взноса при подготовке:
`(SK_ID_CURR, SK_ID_PREV, NUM_INSTALMENT_VERSION, NUM_INSTALMENT_NUMBER)`.
Дата фактической оплаты в него не входит. Суммы частичных оплат складываются точно, а предписанная сумма учитывается один раз. Полные дубли диагностируются; повторение кандидатного ключа не приводит к удалению.

Показываются пропуски всех восьми полей, разбивка по уже существующим split, наличие других оплат того же взноса и сохранность клиентов. Строка без даты фактической оплаты не проходит критерий `<0`; неизвестная сумма не заполняется нулём. Из такой строки нельзя достоверно заключить ни факт оплаты, ни факт неоплаты. В конце проверяется сохранение **числа исходных платёжных строк** и **точной суммы оплат** в агрегированных взносах.

In [13]:
qa_payments=audit_payment_keys(QA_DATA,QA_PREP,QA_REPORT)
display(pd.read_csv(QA_REPORT/'installments_incomplete_key_fields.csv'))
display(pd.read_csv(QA_REPORT/'installments_incomplete_key_patterns.csv'))
display(pd.DataFrame(qa_payments['by_split']))
display(pd.DataFrame(qa_payments['obligation_recovery']))
display(pd.DataFrame(qa_payments['client_retention']))
display(pd.DataFrame([qa_payments['partial_payment_preservation']]).T)
print('Неполных строк:',qa_payments['missing_key_rows'],'Пустые поля:',qa_payments['null_fields'])

,column,part_of_candidate_key,part_of_obligation_key,null_rows,non_null_rows
0,SK_ID_PREV,True,True,0,2905
1,SK_ID_CURR,False,True,0,2905
2,NUM_INSTALMENT_VERSION,True,True,0,2905
3,NUM_INSTALMENT_NUMBER,True,True,0,2905
4,DAYS_INSTALMENT,False,False,0,2905
5,DAYS_ENTRY_PAYMENT,True,False,2905,0
6,AMT_INSTALMENT,False,False,0,2905
7,AMT_PAYMENT,False,False,2905,0


,null_pattern,rows
0,DAYS_ENTRY_PAYMENT; AMT_PAYMENT,2905


,split,incomplete_rows,clients,rows_eligible,rows_excluded
0,development,2091,861,0,2091
1,outside_labeled_applications,322,174,0,322
2,test,492,214,0,492


,split,affected_obligations,obligations_with_other_valid_payments,obligations_without_valid_payments,other_valid_payment_rows,obligations_in_saved_aggregation
0,development,2091,0,2091,0.0,0
1,outside_labeled_applications,322,0,322,0.0,0
2,test,492,0,492,0.0,0


,split,affected_clients,retained_current_applicants,clients_with_other_saved_installment_history
0,development,861,861,856
1,outside_labeled_applications,174,0,0
2,test,214,214,211


,0
raw_rows,13605401
exact_duplicate_row_surplus,0
repeated_candidate_key_groups,770
candidate_key_duplicate_surplus,770
rows_in_repeated_candidate_keys,1540
max_rows_per_candidate_key,2
repeated_keys_with_distinct_payment_amounts,770
eligible_labeled_payment_rows,11589009
eligible_labeled_nonmissing_payment_amounts,11589009
saved_obligations,11024044


Неполных строк: 2905 Пустые поля: ['DAYS_ENTRY_PAYMENT', 'AMT_PAYMENT']


### 8.5. Полностью пустой столбец внутри train-фолда

Даже если во всём development пустых столбцов нет, отдельный train-фолд или 90%-ный повтор может их получить. Для следующего этапа подготовлена фабрика `src.fold_preprocessing.make_numeric_pipeline(scale=False)`; она возвращает **необученный** объект.

Числовая ветка: `SimpleImputer(strategy='median', keep_empty_features=True)`. Если столбец полностью пуст в **текущем train**, медианы не существует: используется заранее заданный технический fallback **0**, столбец сохраняется. Непустые значения этого столбца в validation/test остаются реальными; заполняются только пропуски. Это не оценка по development/test. Для случая без обучающих наблюдений модель не получает оснований оценивать эффект этой переменной.

Параллельно `MissingIndicator(features='all')` создаёт индикатор исходного пропуска для каждого числового столбца, включая новый пропуск в validation. Выходная числовая ширина стабильна — **2 × p** — и не зависит от того, какие пропуски встретились в train. При `scale=True` StandardScaler обучается только в ветке значений, внутри train; индикаторы не масштабируются. После fallback полностью пустого столбца scaler имеет mean=0, scale=1.

Категориальный кодировщик и возможный отбор признаков остаются в полном Pipeline следующего этапа. Списки исходных признаков четырёх уровней сохраняются. Фабрику нужно включить в ColumnTransformer и Pipeline **до передачи в CV**, а не вызвать fit_transform на полном development заранее. Здесь выполняется только проверка на трёх искусственных столбцах; Home Credit не подвергается fit, заполнению, масштабированию или кодированию.

In [14]:
unfitted_numeric_pipeline=make_numeric_pipeline(scale=False)
qa_pipeline=numeric_pipeline_smoke_check(QA_REPORT/'fold_pipeline_checks.json')
print('Демонстрация только на искусственных данных:')
display(pd.DataFrame([qa_pipeline]))

Демонстрация только на искусственных данных:


,scope,sklearn_version,all_checks_passed,raw_numeric_columns,transformed_columns,empty_train_column_fallback,median_for_partly_missing_train_column,validation_observed_value_in_empty_train_column_preserved,new_validation_missing_value_has_indicator,width_is_twice_numeric_column_count,feature_names,fit_policy_ru,fallback_policy_ru,category_policy_ru
0,synthetic_only_no_fit_on_home_credit,1.7.2,True,3,6,0,2,True,True,True,"[values__empty_in_train, values__partly_missin...",fit только на train каждого CV-фолда или 90%-н...,Для полностью пустого train-столбца фиксирован...,Обучаемое кодирование категорий — отдельная ве...


### 8.6. Итог проверки и оставшиеся вопросы

**Уже исправлено на этапе подготовки:** заглушки дат заменены NULL либо поля исключены; неиспользуемые фактические будущие/недатированные платежи отсечены; деление на неположительный знаменатель даёт NULL; частичные оплаты объединены без потери суммы и без повторения номинала взноса. Десятичная арифметика исключает ложную недоплату на границе равенства.

**Сохранено:** редкие большие доходы/суммы, отношения >1, положительные договорные сроки, нулевые суммы отказанных/отменённых прошлых заявок, отрицательные остатки/движения до уточнения их учётного смысла; два признака с постоянным наблюдаемым нулём и пропусками. Качество не означает автоматическую нормализацию распределений.

**Остаётся внутри Pipeline:** медианы по train-фолду, обработка пустого train-столбца с fallback=0 и индикаторами, при необходимости масштабирование, обучаемое кодирование и отбор. Ни один из этих этапов не fit на полном Home Credit.

**Открытые вопросы:** подтверждение дохода 117 млн и единиц/периодов дохода; знак отрицательного первоначального взноса, снятий, лимитов и остатков; неизбежная неполнота сведений о взносах без даты/суммы оплаты; точное время появления обновляемых исторических значений. При недостаточном словаре неоднозначные значения помечены для проверки, а не молча исправлены.

Последняя ячейка подтверждает неизменность всех X, списков признаков, target, fairness, ID и агрегатов; исходные CSV тоже проверяются по SHA-256.

In [15]:
qa_manifest_after=artifact_manifest(QA_PREP)
assert qa_manifest_before==qa_manifest_after,'Подготовленные данные или разбиения изменились!'
assert qa_sources_before==source_manifest(QA_DATA),'Исходные CSV изменились!'
dump_json(QA_REPORT/'prepared_sha256_after.json',qa_manifest_after)
qa_result={'scope':'development_quality_audit_existing_prepared_data',
           'profile':qa_profile,'semantics':qa_semantics,'sentinels':qa_sentinels,
           'payments':qa_payments,'pipeline':qa_pipeline,
           'prepared_files_sha256_unchanged':True,'source_csv_sha256_unchanged':True,
           'X_and_split_files_modified':0,'global_fit_on_home_credit':False,
           'elapsed_seconds':round(time.perf_counter()-QA_STARTED,3)}
dump_json(QA_REPORT/'quality_summary.json',qa_result)
print('Development:',qa_profile['development_rows'],'числовых признаков:',qa_profile['numeric_features'])
print('Строгих нарушений:',qa_semantics['hard_rule_violations'],'нарушений правил отношений:',qa_semantics['ratio_rule_violations'])
print('X, уровни, разбиения и исходники: SHA-256 неизменны. Значения не исправлялись по результатам этого аудита.')
print('Время дополнительного аудита, секунд:',qa_result['elapsed_seconds'])
quality_files=pd.DataFrame([{'file':p.name,'bytes':p.stat().st_size} for p in sorted(QA_REPORT.iterdir()) if p.is_file()])
display(quality_files)

Development: 246008 числовых признаков: 307
Строгих нарушений: 0 нарушений правил отношений: 0
X, уровни, разбиения и исходники: SHA-256 неизменны. Значения не исправлялись по результатам этого аудита.
Время дополнительного аудита, секунд: 165.255


,file,bytes
0,column_development_diagnostics.csv,30381
1,development_levels.csv,181
2,empty_and_constant_columns.csv,350
3,financial_and_ratio_distribution.csv,5361
4,fold_pipeline_checks.json,1246
5,income_tail_review.json,264
6,installments_incomplete_client_retention.csv,181
7,installments_incomplete_key_by_split.csv,155
8,installments_incomplete_key_fields.csv,361
9,installments_incomplete_key_patterns.csv,60
